# Assessing images' compliance to the 60-30-10 rule

## Problem definition

The 60-30-10 rule is a design principle used to help create balenced, cohesive colour schemes by dividing a space into 60% dominant colour, 30% secondary colour, and 10% accent colour. 

An example of it's use could be in interior design where 60% of the room (walls, flooring, large furniture pieces) is made of of the dominant colour, 30% of the room is made up of the secondary colour (curtains, rugs, accent walls, small furniture items), and 10% of the room is made up of the accent colour (throw pillows, art pieces, lampshades, decorations). 

The purpose of this project is to use image clustering algorithms to assess and quantify the level to which different images comply with this rule. 

## Libraries and Setup

Importing required packages

In [1]:
import requests
from PIL import Image
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from skimage import color
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from src.clustering import cluster_pixels
from src.scoring import get_scores

## Functions for Retrieving Images

In [2]:
def get_image_data(source, max_pixels=10000000):
    if source.startswith("http://") or source.startswith("https://"):
        response = requests.get(source)
        img = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        img = Image.open(source).convert("RGB")
    
    imgwidth, imgheight = img.size
    pixelcount = imgwidth * imgheight
    
    if pixelcount > max_pixels:
        scale = 1/np.sqrt(pixelcount / max_pixels)
        imgwidth = max(int(np.floor(imgwidth*scale)), 1)
        imgheight = max(int(np.floor(imgheight*scale)), 1)
        img = img.resize((imgwidth, imgheight), Image.LANCZOS)
        
    return img

In [3]:
imgurl = "https://images.unsplash.com/photo-1765706730202-d8ec6a18ecfa?q=80&w=1471&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
imgurl = "https://images.unsplash.com/photo-1767099181762-17142d05ddcb?q=80&w=774&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
imgurl = "https://images.unsplash.com/photo-1767171361732-c8a90df5c94f?q=80&w=1336&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
imgurl = "https://images.unsplash.com/photo-1767041455614-4e0a5b5cb985?q=80&w=1287&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
imgurl = "https://images.unsplash.com/photo-1766995596065-590702fcce47?q=80&w=1287&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
imgurl = "https://images.unsplash.com/photo-1766343517527-556f58c1c844?q=80&w=1364&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
# imgurl = "https://images.unsplash.com/photo-1766988156039-85f5a592df64?q=80&w=1287&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
# imgurl = "https://images.unsplash.com/photo-1761839257946-4616bcfafec7?q=80&w=1469&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
# imgurl = "https://images.unsplash.com/photo-1767337961735-d6e3e068118f?q=80&w=1287&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
# imgurl = "https://www.elephantstock.com/cdn/shop/articles/choosing-a-color-scheme-60-30-10-rule.jpg?v=1763655192"
# imgurl = "https://cdn.apartmenttherapy.info/image/upload/f_jpg,q_auto:eco,c_fill,g_auto,w_1500,ar_16:9/at%2Fstyle%2F2025-08%2Frule-60-30-10-explained%2Fdecorilla-60-30-10-rule"
# imgurl = "https://avenuerealtygroup.com/wp-content/uploads/2022/10/image003-1-700x467.png"
# imgurl = "https://i0.wp.com/movingtips.wpengine.com/wp-content/uploads/2021/05/color-60-30-10.jpg?fit=400%2C468&ssl=1"
# imgurl = "https://static.standard.co.uk/s3fs-public/thumbnails/image/2019/04/30/16/wes-anderson-moonrise-kingdom.jpg?crop=8:5,smart&quality=75&auto=webp&width=960"
# imgurl = "https://diyvideoeditor.com/wp-content/uploads/wes-anderson-style.jpg"
# imgurl = "https://www.salon.com/app/uploads/2013/10/Screen-Shot-2013-10-17-at-12.11.20-PM.png"

In [ ]:
img = get_image_data(imgurl)
plt.imshow(img)

In [ ]:
img = get_image_data(imgurl, max_pixels=50000)
plt.imshow(img)

In [ ]:
# img = get_image_data("test.png", max_pixels=50000)
plt.imshow(img)

In [ ]:
arr = np.array(img)

In [ ]:
pixels = arr.reshape(-1,3)

In [ ]:
values, counts = np.unique(pixels, axis=0, return_counts=True)

colour_count = 50

# Sort values by counts
idx = np.argsort(counts)[::-1]
values = values[idx]
counts = counts[idx]

# Select top colour_count colors
values = values[:colour_count]
counts = counts[:colour_count]

# Set labels
labels = [f"{np.uint8(colour)}" for colour in values]

# Build plot
plt.figure(figsize=(10, 6))
plt.bar(labels, counts, color=values/255)
plt.xticks(rotation=90)
plt.xlabel("Color (R,G,B)")
plt.ylabel("Pixel Count")
plt.title(f"Top {colour_count} Most Common Colours")
plt.show()

In [ ]:
r = pixels[:,0]
g = pixels[:,1]
b = pixels[:,2]
                        
# Create 3D figure
fig  = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(r, g, b, s=.25, c='blue', alpha=0.25)
ax.set_xlabel('R value')
ax.set_ylabel('G value')
ax.set_zlabel('B value')
ax.set_title("Scatter plot of 3D array")
plt.show()

In [ ]:
plt.subplot(1, 2, 1)
plt.imshow(img)

labels, centers = cluster_pixels(pixels)

centers = np.uint8(centers)
counts = np.bincount(labels)
idx = np.argsort(counts)[::-1]

pixels_clustered = centers[labels].reshape(img.size[1], img.size[0],3)
plt.subplot(1, 2, 2)
plt.imshow(pixels_clustered)
print(get_scores(pixels, labels, centers))

# Set bar labels
bar_labels = [f"{center}" for center in centers[idx]]

# Build plot
plt.figure(figsize=(10, 6))
plt.bar(bar_labels, counts[idx], color=centers[idx]/255)
plt.xticks(rotation=90)
plt.xlabel("Color (R,G,B)")
plt.ylabel("Pixel Count")
plt.title(f"3 cluster centers")
plt.show()

In [ ]:
plt.subplot(1, 2, 1)
plt.imshow(img)

k = 3
kmeans = KMeans(n_clusters=k, n_init=5)
labels = kmeans.fit_predict(pixels)
centers = kmeans.cluster_centers_
centers = np.uint8(centers)
counts = np.bincount(labels)
idx = np.argsort(counts)[::-1]

pixels_kmeans = centers[labels].reshape(img.size[1], img.size[0],3)
plt.subplot(1, 2, 2)
plt.imshow(pixels_kmeans)
# Set bar labels
bar_labels = [f"{center}" for center in centers[idx]]

# Build plot
plt.figure(figsize=(10, 6))
plt.bar(bar_labels, counts[idx], color=centers[idx]/255)
plt.xticks(rotation=90)
plt.xlabel("Color (R,G,B)")
plt.ylabel("Pixel Count")
plt.title(f"3 cluster centers")
plt.show()

In [ ]:
pixels_lab = color.rgb2lab(pixels)

l = pixels_lab[:,0]
a = pixels_lab[:,1]
b = pixels_lab[:,2]

# Create 3D figure
fig  = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(l, a, b, s=.25, c='blue', alpha=0.25)
ax.set_xlabel('L value')
ax.set_ylabel('A value')
ax.set_zlabel('B value')
ax.set_title("Scatter plot of 3D array")
plt.show()

In [ ]:
plt.subplot(1, 2, 1)
plt.imshow(img)

k = 3
kmeans = KMeans(n_clusters=k, n_init=5)
labels = kmeans.fit_predict(pixels_lab)
centers = color.lab2rgb(kmeans.cluster_centers_)
centers = np.uint8(centers*255)
counts = np.bincount(labels)
idx = np.argsort(counts)[::-1]

pixels_kmeans = centers[labels].reshape(img.size[1], img.size[0],3)
plt.subplot(1, 2, 2)
plt.imshow(pixels_kmeans)

# Set bar labels
bar_labels = [f"{center}" for center in centers[idx]]

# Build plot
plt.figure(figsize=(10, 6))
plt.bar(bar_labels, counts[idx], color=centers[idx]/255)
plt.xticks(rotation=90)
plt.xlabel("Color (R,G,B)")
plt.ylabel("Pixel Count")
plt.title(f"{k} cluster centers")
plt.show()

In [ ]:
pixels_hsv = color.rgb2hsv(pixels)

h = pixels_hsv[:,0]
s = pixels_hsv[:,1]
v = pixels_hsv[:,2]

# Create 3D figure
fig  = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(h, s, v, s=.25, c='blue', alpha=0.25)
ax.set_xlabel('H value')
ax.set_ylabel('S value')
ax.set_zlabel('V value')
ax.set_title("Scatter plot of 3D array")
plt.show()

In [ ]:
plt.subplot(1, 2, 1)
plt.imshow(img)

k = 3
kmeans = KMeans(n_clusters=k, n_init=5)
labels = kmeans.fit_predict(pixels_hsv)
centers = color.hsv2rgb(kmeans.cluster_centers_)
centers = np.uint8(centers*255)
counts = np.bincount(labels)
idx = np.argsort(counts)[::-1]

pixels_kmeans = centers[labels].reshape(img.size[1], img.size[0],3)
plt.subplot(1, 2, 2)
plt.imshow(pixels_kmeans)

# Set bar labels
bar_labels = [f"{center}" for center in centers[idx]]

# Build plot
plt.figure(figsize=(10, 6))
plt.bar(bar_labels, counts[idx], color=centers[idx]/255)
plt.xticks(rotation=90)
plt.xlabel("Color (R,G,B)")
plt.ylabel("Pixel Count")
plt.title(f"{k} cluster centers")
plt.show()

## dsgfsdf

In [ ]:
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(img)

k = 3
kmeans = KMeans(n_clusters=k, n_init=5)

# Fit/predict on hue only
hue_values = pixels_hsv[:, 0]
hue_transformed = np.stack([
    np.cos(2 * np.pi * hue_values),
    np.sin(2 * np.pi * hue_values)
], axis=1)
labels = kmeans.fit_predict(hue_transformed)

sil = silhouette_score(hue_transformed, labels)
db = davies_bouldin_score(hue_transformed, labels)
ch = calinski_harabasz_score(hue_transformed, labels)

# Get hue centroids
hue_centers = np.arctan2(
    kmeans.cluster_centers_[:, 1],
    kmeans.cluster_centers_[:, 0]
) / (2 * np.pi) % 1.0  # normalise back to [0, 1]

# Replace each pixel's hue with its cluster's hue centroid, keep original S and V
pixels_hsv_new = pixels_hsv.copy()
pixels_hsv_new[:, 0] = hue_centers[labels]

# Convert back to RGB for image display
pixels_kmeans = np.uint8(color.hsv2rgb(pixels_hsv_new) * 255).reshape(img.size[1], img.size[0], 3)
pixels_hsv_new[:, 1] = 0.8  # saturation
pixels_hsv_new[:, 2] = 0.8  # value
pixels_kmeans_hue = np.uint8(color.hsv2rgb(pixels_hsv_new) * 255).reshape(img.size[1], img.size[0], 3)

plt.subplot(1, 3, 2)
plt.imshow(pixels_kmeans)


plt.subplot(1, 3, 3)
plt.imshow(pixels_kmeans_hue)

# For bar chart: build centers with centroid hue, S=1, V=1 for pure colour swatches
bar_hsv_centers = np.stack([hue_centers, np.full(k, 0.8), np.full(k, 0.8)], axis=1)  # shape (k, 3)
centers = np.uint8(color.hsv2rgb(bar_hsv_centers) * 255)

counts = np.bincount(labels)
idx = np.argsort(counts)[::-1]

percentages = counts[idx] / counts.sum() * 100

bar_labels = [f"{(360 * hue_centers[idx[i]]):.1f}" for i in range(k)]
plt.figure(figsize=(10, 6))
plt.bar(bar_labels, percentages, color=centers[idx] / 255)
plt.xticks(rotation=90)
plt.xlabel("Color (R,G,B)")
plt.ylabel("Percentage (%)")
plt.title(f"{k} cluster centers (hue-only clustering)")
plt.ylim(0, 100)
plt.show()

sil, db, ch

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(5, 5))

# Draw the full colour wheel as a background
theta = np.linspace(0, 2 * np.pi, 360)
for t in theta:
    ax.bar(t, 1, width=2*np.pi/360, color=plt.cm.hsv(t / (2 * np.pi)), linewidth=0)

# Plot each cluster as a marker on the wheel, sized by percentage
for i in range(k):
    angle = hue_centers[idx[i]] * 2 * np.pi
    ax.scatter(angle, 1, s=100, color="white",
               edgecolors='black', linewidth=1.5, zorder=5)
    #ax.annotate(f"{percentages[i]:.1f}%", xy=(angle, 1),
    #            xytext=(angle, 1.2), ha='center', fontsize=10)

ax.set_yticklabels([])
ax.set_xticks(np.linspace(0, 2*np.pi, 12, endpoint=False))
ax.set_xticklabels([f"{int(h)}°" for h in np.linspace(0, 360, 12, endpoint=False)])
ax.set_title("Hue Clusters on Colour Wheel", pad=20)
plt.show()

In [ ]:
def enforce_ratio(labels, hue_transformed, k, ratio=(0.60, 0.30, 0.10)):
    n_pixels = len(labels)
    labels = labels.copy()
    
    # Assign roles by size
    counts = np.bincount(labels, minlength=k)
    
    print("counts:", counts)
    print("size_order:", np.argsort(counts)[::-1])
    print("percentages:", counts / counts.sum() * 100)

    size_order = np.argsort(counts)[::-1]
    
    cluster_to_role = np.zeros(k, dtype=int)
    cluster_to_role[size_order[0]] = 0  # largest -> dominant
    cluster_to_role[size_order[1]] = 1  # second  -> secondary
    cluster_to_role[size_order[2]] = 2  # smallest -> accent
    print("cluster_to_role:", cluster_to_role)  # should be [2, 1, 0]
    print("labels sample before remap:", labels[:10])
    labels = cluster_to_role[labels]
    print("labels sample after remap:", labels[:10])
    print("counts after remap:", np.bincount(labels, minlength=3))
    
    # Compute cluster centers in transformed space (after role remapping)
    centers = np.stack([
        hue_transformed[labels == role].mean(axis=0) for role in range(3)
    ])
    
    # Compute distance from every pixel to every role center
    # shape: (n_pixels, 3)
    dist_matrix = np.linalg.norm(
        hue_transformed[:, np.newaxis, :] - centers[np.newaxis, :, :], axis=2
    )
    
    # Target counts
    targets = np.array([int(r * n_pixels) for r in ratio])
    targets[-1] = n_pixels - targets[:-1].sum()
    
    # New labels array to fill
    new_labels = np.full(n_pixels, -1, dtype=int)
    
    # For each role, sort all pixels by distance to that role's center
    # Fill each role's quota with the closest available (unassigned) pixels
    role_fill_order = [0, 1, 2]  # fill dominant first, then secondary, then accent

    print("targets:", targets)  # should be [~29935, ~14967, ~4984] for 60/30/10
    print("counts before enforcement:", np.bincount(labels, minlength=3))
    
    for role in role_fill_order:
        # Sort all unassigned pixels by distance to this role's center
        unassigned = np.where(new_labels == -1)[0]
        dists = dist_matrix[unassigned, role]
        closest = unassigned[np.argsort(dists)]
        # Assign the closest n pixels up to this role's target
        quota = targets[role]
        new_labels[closest[:quota]] = role

    print("final counts:", np.bincount(new_labels, minlength=3))
    print("final percentages:", np.bincount(new_labels, minlength=3) / len(new_labels) * 100)
        
    # Assign any remaining unassigned pixels to accent (rounding leftover)
    leftover = np.where(new_labels == -1)[0]
    new_labels[leftover] = 2
    
    return new_labels, targets, size_order

In [ ]:
labels_enforced, targets, size_order = enforce_ratio(labels, hue_transformed, k=3)

In [ ]:
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(img)

sil = silhouette_score(hue_transformed, labels_enforced)
db = davies_bouldin_score(hue_transformed, labels_enforced)
ch = calinski_harabasz_score(hue_transformed, labels_enforced)

# Get hue centroids
hue_centers = np.arctan2(
    kmeans.cluster_centers_[:, 1],
    kmeans.cluster_centers_[:, 0]
) / (2 * np.pi) % 1.0  # normalise back to [0, 1]

# Replace each pixel's hue with its cluster's hue centroid, keep original S and V
pixels_hsv_new = pixels_hsv.copy()
hue_centers_remapped = hue_centers[size_order]
pixels_hsv_new[:, 0] = hue_centers_remapped[labels_enforced]

# Convert back to RGB for image display
pixels_kmeans = np.uint8(color.hsv2rgb(pixels_hsv_new) * 255).reshape(img.size[1], img.size[0], 3)
pixels_hsv_new[:, 1] = 0.8  # saturation
pixels_hsv_new[:, 2] = 0.8  # value
pixels_kmeans_hue = np.uint8(color.hsv2rgb(pixels_hsv_new) * 255).reshape(img.size[1], img.size[0], 3)

plt.subplot(1, 3, 2)
plt.imshow(pixels_kmeans)


plt.subplot(1, 3, 3)
plt.imshow(pixels_kmeans_hue)

# For bar chart: build centers with centroid hue, S=1, V=1 for pure colour swatches
bar_hsv_centers = np.stack([hue_centers_remapped, np.full(k, 0.8), np.full(k, 0.8)], axis=1)  # shape (k, 3)
centers = np.uint8(color.hsv2rgb(bar_hsv_centers) * 255)

counts = np.bincount(labels_enforced)
idx = np.argsort(counts)[::-1]

percentages = counts[idx] / counts.sum() * 100

bar_labels = [f"{(360 * hue_centers_remapped[idx[i]]):.1f}" for i in range(k)]
plt.figure(figsize=(10, 6))
plt.bar(bar_labels, percentages, color=centers[idx] / 255)
plt.xticks(rotation=90)
plt.xlabel("Color (R,G,B)")
plt.ylabel("Percentage (%)")
plt.title(f"{k} cluster centers (hue-only clustering)")
plt.ylim(0, 100)
plt.show()